<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/main/notebooks/01-processamento_pln.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

📑 Guia de Execução Estratégica
⚠️ IMPORTANTE: Sempre que o Runtime (Ambiente de Execução) for reiniciado, as Células 1 e 2 devem ser executadas obrigatoriamente para restabelecer os caminhos do Drive e reinstalar as bibliotecas.

🔄 Fluxo de Dependências:
Sessão Recém-Iniciada: Executar Célula 1 ➔ Célula 2.

Primeira vez no projeto: Executar Célula 1 ➔ Célula 2 ➔ Célula 3 (Carga).

Retomando Processamento: Se o banco já existe no Drive, pule a Célula 3 e vá direto para a Célula 4 e/ou 5 e/ou 6.

In [ ]:
# Célula 1: Montagem do Google Drive e Configuração de Caminhos
from google.colab import drive
import os

# 1. Montagem Segura: Só executa se ainda não estiver montado
if not os.path.exists('/content/drive/MyDrive'):
    print("📂 Montando Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive já está montado e acessível.")

# 2. Configuração Estrita de Caminhos
DRIVE_DIR = '/content/drive/MyDrive/mba-engsof-tcc/versao_pos_entrega'
DB_FILE_NAME = 'data/base-dados.db'
DB_PATH = os.path.join(DRIVE_DIR, DB_FILE_NAME)

# Artefatos SQL
SCHEMA_SQL = os.path.join(DRIVE_DIR, 'sql/01-schema.sql')
SEED_SQL = os.path.join(DRIVE_DIR, 'sql/02-seed_data.sql')

# Pasta de Saída (Outputs)
EXPORT_PATH = os.path.join(DRIVE_DIR, 'outputs')
if not os.path.exists(EXPORT_PATH):
    os.makedirs(EXPORT_PATH)
    print(f"📁 Pasta de exportação criada em: {EXPORT_PATH}")

print(f"📍 Banco de Dados: {DB_PATH}")

In [ ]:
# Célula 2: Instalação das bibliotecas e inicialização da estrutura (Schema)

# 1. Instalação Silenciosa
!pip install -q transformers torch pandas bertopic pysentimiento spacy
!python -m spacy download pt_core_news_lg -q

import sqlite3
import torch

# 2. Hardware Check para BERTimbau/BERTopic
device = 0 if torch.cuda.is_available() else -1

def inicializar_estrutura_db(db_path, schema_path):
    """Garante que a estrutura de tabelas esteja presente."""
    print(f"🛠️ Verificando integridade das tabelas...")

    # Se o arquivo de banco não existir, o SQLite o criará automaticamente
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        with open(schema_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())
        conn.commit()
        print("✅ Estrutura (Schema) validada com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao processar Schema: {e}")
    finally:
        conn.close()

# 3. Execução
inicializar_estrutura_db(DB_PATH, SCHEMA_SQL)

print(f"\n🚀 Ambiente pronto (GPU: {'Ativa' if device == 0 else 'Inativa'}).")

In [ ]:
# Célula 3: Carga Inicial de Dados (Seed SQL)
def executar_carga_dados(db_path, seed_path):
    """Popula o banco apenas se a tabela 'verso' estiver vazia."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        # Verifica se já existem dados para evitar duplicidade no Drive
        cursor.execute("SELECT count(*) FROM verso")
        total_existente = cursor.fetchone()[0]

        if total_existente > 0:
            print(f"ℹ️ O banco já contém {total_existente} versos. Carga inicial ignorada.")
            return

        print("🌱 Semeando dados iniciais (02-seed_data.sql)... Isso pode levar alguns minutos.")
        with open(seed_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())

        conn.commit()
        print(f"✅ Carga de {seed_path} concluída com sucesso!")

    except sqlite3.OperationalError as e:
        print(f"⚠️ Erro operacional: {e}. Verifique se a Célula 2 foi executada.")
    except Exception as e:
        print(f"❌ Erro crítico na carga: {e}")
    finally:
        conn.close()

# Executa a carga (Somente se necessário)
executar_carga_dados(DB_PATH, SEED_SQL)

In [ ]:
# Célula 4: Indexação Profunda e Extração de Triplas Sintáticas
import spacy
import sqlite3
import pandas as pd
import numpy as np

# 1. Preparação do Modelo
try:
    nlp = spacy.load("pt_core_news_lg")
except:
    !python -m spacy download pt_core_news_lg
    nlp = spacy.load("pt_core_news_lg")

def executar_indexacao_profunda(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Carregando os versos (Pode filtrar por genero_id se desejar testar)
    df_versos = pd.read_sql_query("SELECT id, texto FROM verso", conn)

    print(f"🧠 Processando {len(df_versos)} versos com Hierarquia Sintática...")

    # Limpeza da tabela verso_limpo (as outras limpamos no SQL acima)
    cursor.execute("DELETE FROM verso_limpo")

    for _, row in df_versos.iterrows():
        verso_id = row['id']
        texto = row['texto']

        if not texto or len(texto.strip()) < 3:
            continue

        doc = nlp(texto)
        total_tokens = len(doc)

        # --- A. COLETA DE MÉTRICAS DO VERSO (Nível Macro) ---
        counts = {'ADJ': 0, 'ADV': 0, 'PROPN': 0, 'VERB': 0, 'NUM': 0, 'NOUN': 0, 'PRON': 0}
        n_primeira_pessoa = 0
        palavras_sig_len = []

        # --- B. INDEXAÇÃO DE PALAVRAS E HIERARQUIA (Nível Micro) ---
        for t in doc:
            # 1. Estatísticas para o verso
            if t.pos_ in counts:
                counts[t.pos_] += 1
            if t.morph.get("Person") == ["1"]:
                n_primeira_pessoa += 1
            if t.pos_ in ['NOUN', 'ADJ'] and not t.is_stop:
                palavras_sig_len.append(len(t.text))

            # 2. Gestão do Dicionário (Tabela 'palavra')
            cursor.execute("""
                INSERT OR IGNORE INTO palavra (lemma, pos_tag, is_stop)
                VALUES (?, ?, ?)
            """, (t.lemma_.lower(), t.pos_, 1 if t.is_stop else 0))

            cursor.execute("SELECT id FROM palavra WHERE lemma = ?", (t.lemma_.lower(),))
            palavra_id = cursor.fetchone()[0]

            # 3. Vínculo Hierárquico (Tabela 'verso_palavra')
            cursor.execute("""
                INSERT INTO verso_palavra (
                    verso_id, palavra_id, posicao, head_pos, dep_relation, morph
                ) VALUES (?, ?, ?, ?, ?, ?)
            """, (
                verso_id,
                palavra_id,
                t.i,
                t.head.i,
                t.dep_,
                str(t.morph) # Salva como string formatada para consulta posterior
            ))

        # --- C. CÁLCULO FINAL E PERSISTÊNCIA DO VERSO ---
        score_emocional = (counts['ADJ'] + counts['ADV']) / total_tokens
        score_informativo = (counts['PROPN'] + counts['NUM']) / total_tokens
        score_acao = counts['VERB'] / total_tokens
        avg_word_len = np.mean(palavras_sig_len) if palavras_sig_len else 0.0

        present_tags = [v for v in counts.values() if v > 0]
        entropia = -sum([(v/total_tokens) * np.log(v/total_tokens + 1e-9) for v in present_tags])

        texto_limpo = " ".join([t.text.lower() for t in doc if not t.is_stop and not t.is_punct])

        cursor.execute("""
            INSERT INTO verso_limpo (
                verso_id, texto_limpo, score_emocional, score_informativo,
                score_acao, entropia_gramatical, n_primeira_pessoa, avg_word_len
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (verso_id, texto_limpo, score_emocional, score_informativo,
              score_acao, entropia, n_primeira_pessoa, avg_word_len))

    conn.commit()
    conn.close()
    print("✅ Indexação Profunda Concluída com Sucesso!")

# Chamada da função
executar_indexacao_profunda(DB_PATH)

In [ ]:
# Célula 5: Classificação por Eixos Existenciais
import sqlite3
import pandas as pd
import numpy as np
from transformers import pipeline
from tqdm.notebook import tqdm

# 1. Configuração e Carga de Dados
conn = sqlite3.connect(DB_PATH)

# Recupera os Eixos e as Descrições Concatenadas do Banco
query_eixos = """
    SELECT e.id, e.nome, GROUP_CONCAT(ed.sentenca, ' ') as descricao_completa
    FROM eixo e
    JOIN eixo_descricao ed ON e.id = ed.eixo_id
    GROUP BY e.id
    ORDER BY e.id
"""
df_eixos = pd.read_sql_query(query_eixos, conn)
# Lista de labels (descrições) que a IA usará para classificar
labels_ia = df_eixos['descricao_completa'].tolist()
mapeamento_eixos = df_eixos['nome'].tolist()

# Recupera os Versos com Metadados XAI da Célula 4
query_versos = """SELECT vl.* FROM verso_limpo vl
                  JOIN verso v on v.id = vl.verso_id
                  JOIN livro l on l.id = v.livro_id
                  WHERE l.abreviacao = 'Fp'"""
df_input = pd.read_sql_query(query_versos, conn)

# 2. Inicialização do Modelo Zero-Shot
# O BART-Large-MNLI é ideal para entender descrições detalhadas de eixos
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli",
                      device=0) # 0 para GPU, -1 para CPU

def classificar_com_dna(row, labels):
    # --- CONSTRUÇÃO DO PROMPT ENRIQUECIDO (DNA SINTÁTICO) ---
    dna = []
    if row['n_primeira_pessoa'] > 0: dna.append("Relato Pessoal/Subjetivo")
    if row['is_identidade'] == 1: dna.append("Definição de Identidade/Estado")
    if row['score_emocional'] > 0.2: dna.append("Alta Carga Emocional")
    if row['tem_numeral'] == 1: dna.append("Dados Quantitativos/Inventário")
    if row['avg_word_len'] > 7: dna.append("Vocabulário Abstrato/Complexo")

    prefixo = f"[Contexto: {', '.join(dna)}] " if dna else ""
    # Enviamos o texto original ou limpo precedido pelo DNA sintático
    texto_para_ia = prefixo + row['texto_limpo']

    # Classificação Zero-Shot
    res = classifier(texto_para_ia, labels, multi_label=False)

    # Organiza os scores na ordem original dos IDs dos eixos (0, 1, 2, 3...)
    scores_ordenados = [res['scores'][res['labels'].index(l)] for l in labels]
    return scores_ordenados

# 3. Processamento
print(f"🤖 Classificando {len(df_input)} versos...")
results = []

for _, row in tqdm(df_input.iterrows(), total=len(df_input)):
    probs = classificar_com_dna(row, labels_ia)

    # Cálculos de Governança (Entropia e Gap)
    probs_sorted = sorted(probs, reverse=True)
    gap = probs_sorted[0] - probs_sorted[1]
    entropia = -sum([p * np.log(p + 1e-9) for p in probs])

    idx_vencedor = np.argmax(probs)
    similaridade = probs[idx_vencedor]

    # Status de Decisão baseado nos Metadados + IA
    status = "Contexto Aumentado"
    if gap > 0.4: status = "Alta Confiança"
    if entropia > 1.0: status = "Ambiguidade Controlada"
    if row['tem_numeral'] == 1 and idx_vencedor == 3: status = "Validado por Dados (XAI)"

    results.append({
        'verso_id': int(row['verso_id']),
        'topico_id': idx_vencedor,
        'p_exaustao': probs[0],
        'p_transitoriedade': probs[1],
        'p_vazio': probs[2],
        'p_narrativo': probs[3],
        'similaridade_final': similaridade,
        'margem_dominancia': gap,
        'status_decisao': status,
        'entropia': entropia,
        'gap_confianca': gap
    })

# 4. Persistência na Tabela Existente
df_final = pd.DataFrame(results)
cursor = conn.cursor()
cursor.execute("DELETE FROM verso_topico") # Limpa resultados anteriores

df_final.to_sql('verso_topico', conn, if_exists='append', index=False)

conn.commit()
conn.close()
print("✨ Classificação concluída com sucesso e armazenada em 'verso_topico'!")

In [ ]:
# Célula 6: Análise de Sentimento Contextual e Cruzamento Existencial
from pysentimiento import create_analyzer
import pandas as pd
import sqlite3
from tqdm.auto import tqdm

# 1. Inicializar o Analisador
print("🚀 Carregando modelo Transformer para Sentimento (PT-BR)...")
# O analisador 'sentiment' para 'pt' é baseado em BERTimbau, ideal para o TCC
analyzer = create_analyzer(task="sentiment", lang="pt")

# 2. Busca do texto original e dos tópicos
conn = sqlite3.connect(DB_PATH)
df_input = pd.read_sql_query("""
    SELECT v.id as verso_id, v.texto, vt.topico_id
    FROM verso v
    JOIN verso_topico vt ON v.id = vt.verso_id
""", conn)

textos = df_input['texto'].tolist()
verso_ids = df_input['verso_id'].tolist()

# 3. Execução da análise em lotes (Aproveitando a GPU se disponível)
print(f"📊 Analisando carga emocional de {len(textos)} versículos...")
sentimentos = []
batch_size = 64
mapa_num = {'POS': 1, 'NEU': 0, 'NEG': -1}

# O predict em lote é significativamente mais rápido no Colab
for i in tqdm(range(0, len(textos), batch_size)):
    lote = textos[i:i + batch_size]
    ids_lote = verso_ids[i:i + batch_size]
    preds_lote = analyzer.predict(lote)

    for idx, p in enumerate(preds_lote):
        # Capturamos as probabilidades brutas para análises de incerteza se necessário
        sentimentos.append({
            'verso_id': ids_lote[idx],
            'label': p.output,
            'sentimento_num': mapa_num.get(p.output, 0),
            'score_pos': p.probas.get('POS', 0),
            'score_neg': p.probas.get('NEG', 0),
            'score_neu': p.probas.get('NEU', 0)
        })

df_sent = pd.DataFrame(sentimentos)

# 4. Persistência dos Resultados
try:
    cursor = conn.cursor()
    # Limpamos para garantir que a nova classificação da Célula 5 seja a única presente
    cursor.execute("DELETE FROM verso_sentimento")

    # Inserimos os novos resultados (Integridade referencial com 'verso_id')
    df_sent.to_sql('verso_sentimento', conn, if_exists='append', index=False)
    conn.commit()
    print("\n✅ Célula 6 concluída! Sentimentos processados e salvos com sucesso.")

    # 5. RESULTADO FINAL: O DIAGNÓSTICO (PROBLEMA) VS. A CURA (ANTÍDOTO)
    print("\n📈 RESUMO EXECUTIVO: PROBLEMÁTICA (CRISE) VS. ANTÍDOTO (CURA)")

    res_final = pd.read_sql_query("""
        SELECT
            t.antidoto_referencia as Eixo_Filosofico,
            COUNT(*) as Total_Versos,
            SUM(CASE WHEN vs.sentimento_num = 1 THEN 1 ELSE 0 END) as Antidotos_Cura,
            SUM(CASE WHEN vs.sentimento_num = -1 THEN 1 ELSE 0 END) as Problematica_Crise,
            ROUND(AVG(vs.sentimento_num), 3) as Polaridade_Media
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
        WHERE t.id != 3 -- Foco nos eixos Han, Bauman e Frankl
        GROUP BY t.antidoto_referencia
        ORDER BY Polaridade_Media DESC
    """, conn)

    # Exibe a tabela formatada no Colab
    display(res_final)

except Exception as e:
    print(f"❌ Erro na persistência: {e}")
finally:
    conn.close()